In [1]:
import os
print(os.getcwd())

/Users/matt/Desktop/Patent Litigation Analysis


In [3]:
#importing libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re as re

pacer = pd.read_csv('/Users/matt/Desktop/Patent Litigation Analysis/data/processed/pacer_cases_processed_v2.csv')
cases = pd.read_csv('data/processed/cases_processed_v3.csv')

In [15]:
# Remove rows where case_number_clean starts with "font"
pacer = pacer[~pacer['case_number_clean'].str.startswith('font', na=False)]
pacer = pacer[~pacer['case_number_clean'].str.startswith('b', na=False)]

In [19]:
cases['case_number_clean'] = cases['case_number_clean'].str.strip()
pacer['case_number_clean'] = pacer['case_number_clean'].str.strip()
cases['case_name'] = cases['case_name'].str.strip()
pacer['case_name'] = pacer['case_name'].str.strip()

cases_unmatched_mask = ~cases['case_number_clean'].isin(pacer['case_number_clean'])
pacer_unmatched_mask = ~pacer['case_number_clean'].isin(cases['case_number_clean'])

cases_unmatched = cases.loc[cases_unmatched_mask, ['case_name', 'case_number_clean']].copy()
pacer_unmatched = pacer.loc[pacer_unmatched_mask, ['case_name', 'case_number_clean']].copy()

cases_name_to_number = (
    cases_unmatched
    .drop_duplicates(subset='case_name')
    .set_index('case_name')['case_number_clean']
)

match_mask = pacer.loc[pacer_unmatched_mask, 'case_name'].isin(cases_name_to_number.index)

print("PACER unmatched rows with exact case_name match in CASES unmatched:", match_mask.sum())

pacer.loc[pacer_unmatched_mask & pacer['case_name'].isin(cases_name_to_number.index), 'case_number_clean'] = (
    pacer.loc[pacer_unmatched_mask & pacer['case_name'].isin(cases_name_to_number.index), 'case_name']
    .map(cases_name_to_number)
)

print("Done updating PACER case_number_clean where unmatched case_name matched exactly.")

PACER unmatched rows with exact case_name match in CASES unmatched: 25
Done updating PACER case_number_clean where unmatched case_name matched exactly.


In [20]:
cases['case_number_clean'] = cases['case_number_clean'].str.strip()
pacer['case_number_clean'] = pacer['case_number_clean'].str.strip()

cases_matched = cases['case_number_clean'].isin(
    pacer['case_number_clean']
)

pacer_matched = pacer['case_number_clean'].isin(
    cases['case_number_clean']
)

print("CASES → PACER")
print("Matched:", cases_matched.sum())
print("Unmatched:", (~cases_matched).sum())
print("Total:", len(cases))

print("\nPACER → CASES")
print("Matched:", pacer_matched.sum())
print("Unmatched:", (~pacer_matched).sum())
print("Total:", len(pacer))

CASES → PACER
Matched: 74628
Unmatched: 1
Total: 74629

PACER → CASES
Matched: 74671
Unmatched: 180
Total: 74851


In [21]:
cases['case_number_clean'] = cases['case_number_clean'].str.strip()
pacer['case_number_clean'] = pacer['case_number_clean'].str.strip()

pacer_unmatched_mask = ~pacer['case_number_clean'].isin(
    cases['case_number_clean']
)

pacer_unmatched = pacer[pacer_unmatched_mask]

print("Total unmatched PACER rows:", len(pacer_unmatched))

print(pacer_unmatched['case_number_clean'])

Total unmatched PACER rows: 180
229      9:2013-cv-80395
235      9:2013-cv-80113
320      9:2011-cv-80123
3468     8:1991-cv-00662
8210     5:2014-cv-05206
              ...       
73015    1:1983-cv-03126
73163    0:2014-cv-62367
73260    0:2013-cv-61681
73430    0:2011-mc-60079
73447    0:2011-cv-60363
Name: case_number_clean, Length: 180, dtype: str


In [22]:
cases_unmatched_mask = ~cases['case_number_clean'].isin(
    pacer['case_number_clean']
)
cases_unmatched = cases[cases_unmatched_mask]
print("Total unmatched CASES rows:", len(cases_unmatched))
print(cases_unmatched['case_number_clean'])

Total unmatched CASES rows: 1
18710    4:2015-cv-05487
Name: case_number_clean, dtype: str


In [24]:
pacer.to_csv('/Users/matt/Desktop/Patent Litigation Analysis/data/processed/pacer_cases_processed_v4.csv', index=False)